In [ ]:
!pip  install geopy pandas googlemaps

  Preparing metadata (setup.py) ... done
  Created wheel for googlemaps: filename=googlemaps-4.10.0-py3-none-any.whl size=40714 sha256=c163dac2361c940cc488e8d63df3db24eda25c891b93fc5a38cc4d5993b405a2
  Stored in directory: /root/.cache/pip/wheels/f1/09/77/3cc2f5659cbc62341b30f806aca2b25e6a26c351daa5b1f49a
Successfully built googlemaps


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import googlemaps
import json
import time

GOOGLE_MAPS_API_KEY = ""
UNIVERSITY         = "University of California,Irvine"
RADIUS_MILES       = 150
SEARCH_KEYWORD     = "manufacturer"

geolocator   = Nominatim(user_agent="whisper_energy_locator")
loc          = geolocator.geocode("University of California, Irvine")
univ_coords  = (loc.latitude, loc.longitude)
gmaps        = googlemaps.Client(key=GOOGLE_MAPS_API_KEY)
RADIUS_M  = 150 * 1609.34
keyword   = "manufacturer"

all_places = []
resp = gmaps.places_nearby(
    location=univ_coords,
    radius=RADIUS_M,
    keyword=keyword,
    type="establishment"
)
all_places.extend(resp.get("results", []))

while "next_page_token" in resp:
    # Google requires a short delay before the next page token becomes valid
    time.sleep(2)
    resp = gmaps.places_nearby(page_token=resp["next_page_token"])
    all_places.extend(resp.get("results", []))

print(f"Found {len(all_places)} total places in radius")




Found 60 total places in radius


In [ ]:
with open("all_manufacturers.json", "w", encoding="utf8") as fp:
    json.dump(all_places, fp, indent=2, ensure_ascii=False)
print("Wrote full results to all_manufacturers.json")



Wrote full results to all_manufacturers.json


In [ ]:
import time
import pandas as pd
import googlemaps
from geopy.distance import geodesic

DETAIL_FIELDS = [
    "website", "url", "formatted_phone_number", "business_status",
    "formatted_address", "rating", "user_ratings_total", "opening_hours"
]

rows = []
errors = []

for i, place in enumerate(all_places, 1):
    pid = place.get("place_id")
    try:
        detail = gmaps.place(place_id=pid, fields=DETAIL_FIELDS)["result"]
    except Exception as e:
        errors.append((pid, str(e)))
        print(f"[{i}/{len(all_places)}] ⚠️ {pid} error: {e}")
        # If you hit OVER_QUERY_LIMIT, you might need a longer sleep or fewer fields
        if "OVER_QUERY_LIMIT" in str(e):
            print("  → Sleeping for 10s to back off…")
            time.sleep(10)
            continue
        else:
            continue

    # Compute distance if you still need it…
    locn = place["geometry"]["location"]
    dist_mi = geodesic(univ_coords, (locn["lat"], locn["lng"])).meters / 1609.34

    rows.append({
        "place_id":          pid,
        "name":              detail.get("name") or place.get("name"),
        "vicinity":          place.get("vicinity"),
        "formatted_address":detail.get("formatted_address"),
        "latitude":          locn["lat"],
        "longitude":         locn["lng"],
        "distance_miles":    round(dist_mi,2),
        "types":             ",".join(place.get("types", [])),
        "business_status":   detail.get("business_status"),
        "rating":            detail.get("rating"),
        "user_ratings_total":detail.get("user_ratings_total"),
        "phone":             detail.get("formatted_phone_number"),
        "website":           detail.get("website"),
        "google_url":        detail.get("url"),
        "opening_hours":     detail.get("opening_hours",{}).get("weekday_text", [])
    })

    print(f"[{i}/{len(all_places)}] ✅ {pid} fetched")
    time.sleep(0.2)  # even a 0.2s pause helps avoid spikes

print(f"\nDone. Successfully fetched {len(rows)} details; {len(errors)} errors.")
df_details = pd.DataFrame(rows)





[1/60] ✅ ChIJF6WiGvq33IARy4CDVfy-Kkk fetched
[2/60] ✅ ChIJWxHGsNzI3IAReRNxO4D_YLI fetched
[3/60] ✅ ChIJizLeOHjW3IARkYtZtkXq3Hk fetched
[4/60] ✅ ChIJlfvcVvgl3YARNjoyVnAGB3A fetched
[5/60] ✅ ChIJKT0vUbKz3IARD3j5UQNDVJI fetched
[6/60] ✅ ChIJTed-3znh3IARjHohKKl3f68 fetched
[7/60] ✅ ChIJ6Xhow8Qo3YAR5kw4mhKrc3c fetched
[8/60] ✅ ChIJJ1y0-B7X3IARHZO2mKnZHFg fetched
[9/60] ✅ ChIJ4xs5B5PQwoAR7daHO95T3ks fetched
[10/60] ✅ ChIJI9VjWNUr3YARcWNGpjBL1bo fetched
[11/60] ✅ ChIJMQIhJXPc3IARG6J1lZRxwEM fetched
[12/60] ✅ ChIJnwTffosu3YARz9eYp5n16mo fetched
[13/60] ✅ ChIJMQmKDG3KwoARYe6i99cFSsU fetched
[14/60] ✅ ChIJFy4485Qs3YARtFNu51f2hwg fetched
[15/60] ✅ ChIJCQSyIP3e3IARG0BXNtBp0C8 fetched
[16/60] ✅ ChIJHXzK5bkyw4AR0tGiF59Fxlw fetched
[17/60] ✅ ChIJZVLpIv3p3IARC175gVf3nmY fetched
[18/60] ✅ ChIJjY0MC05Lw4ARsRgggVw_nWQ fetched
[19/60] ✅ ChIJeYjTzw7W3IARu2bqPA_-gj0 fetched
[20/60] ✅ ChIJa6YgZCjZ3IARxOFwF5axurg fetched
[21/60] ✅ ChIJQVOOV47W3IARePC915QHEPM fetched
[22/60] ✅ ChIJN0_AxjUn3YARYRFVExZ3xag fetch

In [ ]:
out_path = "UCImanufacturers_details.csv"
df_details.to_csv(out_path, index=False)
print(f"Wrote all place details to {out_path}")

Wrote all place details to UCImanufacturers_details.csv


*HIFLD*